In [4]:
Sys.setenv(
  http_proxy  = "http://100.83.45.1:7897",
  https_proxy = "http://100.83.45.1:7897",
  all_proxy   = "socks5://100.83.45.1:7897",
  no_proxy    = "localhost,127.0.0.1"
)
options(timeout = 3600)

↑ 如果直接尝试下载，会因为网络原因失败，疑似需要代理

本机暂未部署代理，但加入了另外几台机器的 tailscale 网络，而其中一台机器有代理（100.83.45.1:7897），本次使用该机器的代理下载数据（在notebook顶端设置）

In [5]:
library(spatialLIBD)
library(ExperimentHub)

In [6]:
ehub <- ExperimentHub()
spe <- fetch_data(type = "spe", eh = ehub)

adding rname 'https://www.dropbox.com/s/f4wcvtdq428y73p/Human_DLPFC_Visium_processedData_sce_scran_spatialLIBD.Rdata?dl=1'



2026-10-14 13:47:49.969326 loading file /home/liuzy/.cache/R/BiocFileCache/1712f3e97c3c3_Human_DLPFC_Visium_processedData_sce_scran_spatialLIBD.Rdata%3Fdl%3D1



In [13]:
table(spe$sample_id)


151507 151508 151509 151510 151669 151670 151671 151672 151673 151674 151675 
  4226   4384   4789   4634   3661   3498   4110   4015   3639   3673   3592 
151676 
  3460 

In [15]:
#! 注意，旧 spatialLIBD 文件，如作者提供的 sample_151507_anno.h5，和本次新下载的 spatialLIBD 文件在细节方面不一致
sel <- spe$sample_id == "151507"
array_pos <- cbind(spe$array_row[sel], spe$array_col[sel]) # 网格索引（整数）
pixel_pos <- SpatialExperiment::spatialCoords(spe)[sel, ] # 像素坐标（浮点）

range(array_pos); head(array_pos)
range(pixel_pos); head(pixel_pos)

[1]   0 127

0,16
50,102
3,43
59,19
43,9
47,13


[1]  2160 11751

,pxl_col_in_fullres,pxl_row_in_fullres
AAACAACGAATAGTTC-1,3276,2514
AAACAAGTATCTCCCA-1,9178,8520
AAACAATCTACTAGCA-1,5133,2878
AAACACCAATAACTGC-1,3462,9581
AAACAGCTTTCAGAAG-1,2779,7663
AAACAGGGTCTATATT-1,3053,8143


In [17]:
library(rhdf5)
f <- "../../data/sample_151507_anno.h5"
Pos <- h5read(f, "Pos")   # (4226,2) 整数网格
Loc <- h5read(f, "Loc")   # (4226,2) 像素
range(Pos); range(Loc)
par(mfrow = c(1, 2))
plot(Pos, pch = 19, cex = 0.3, asp = 1, main = "Pos (array)")
plot(Loc, pch = 19, cex = 0.3, asp = 1, main = "Loc (pixel)")

ERROR: Error in H5Fopen(file, flags = flags, fapl = fapl, native = native): HDF5. File accessibility. Unable to open file.


In [18]:
# 将 spe 按 sample_id 拆成 12 个 DSSC 输入格式的 h5
# X: spots x genes 计数; Y: 真值标签(Layer1..6/WM/NA); Pos: spots x 2; Gene: 基因名
outdir <- "../../data/spatialLIBD"
dir.create(outdir, showWarnings = FALSE, recursive = TRUE)
cnt <- SummarizedExperiment::assay(spe, "counts")
genes <- SummarizedExperiment::rowData(spe)$gene_name
coord <- SpatialExperiment::spatialCoords(spe)
lab <- as.character(spe$layer_guess_reordered)
ids <- spe$sample_id
for (s in unique(ids)) {
    i <- which(ids == s)
    # X <- t(as.matrix(cnt[, i, drop = FALSE]))
    X <- as.matrix(cnt[, i, drop = FALSE]) 
    Y <- lab[i]; Y[is.na(Y)] <- "NA"
    # Pos <- coord[i, , drop = FALSE]
    Pos <- cbind(spe$array_row[i], spe$array_col[i]) #! R: spots x 2  -> h5py: (2, spots)
    fp <- file.path(outdir, paste0("sample_", s, "_anno.h5"))
    if (file.exists(fp)) file.remove(fp)
    rhdf5::h5createFile(fp)
    rhdf5::h5write(X, fp, "X")
    rhdf5::h5write(Y, fp, "Y")
    rhdf5::h5write(Pos, fp, "Pos")
    rhdf5::h5write(genes, fp, "Gene")
    message("wrote ", fp, "  X=", paste(dim(X), collapse = "x"))
}

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.1 GiB”
You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the dataset, you should test smaller chunk sizes to improve read times.

wrote ../../data/spatialLIBD/sample_151507_anno.h5  X=33538x4226

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.1 GiB”
You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the dataset, you should test smaller chunk sizes to improve read times.

wrote ../../data/spatialLIBD/sample_151508_anno.h5  X=33538x4384

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.2 GiB”
You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the data